# Reintegration readiness — resident-level prediction

Harbor of Hope case management.


## 1. Business Understanding

**PREDICTIVE.** We estimate **reintegration readiness** (binary `reintegration_ready` from the master builder: completed reintegration **or** improved risk level). This supports **prioritizing case planning** — not establishing causal efficacy of a single factor.

**Success:** ROC-AUC ≥ **0.60** on the test set; qualitative review of errors for high-stakes cases.


## 2. Data Understanding & Preparation (EDA)


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
import warnings; warnings.filterwarnings('ignore')

from pathlib import Path
import subprocess, sys, warnings
warnings.filterwarnings("ignore")

def _find_ml_dir() -> Path:
    p = Path.cwd().resolve()
    for _ in range(10):
        b = p / "build_master_datasets.py"
        d = p / "data" / "supporters.csv"
        if b.exists() and d.exists():
            return p
        v2 = p / "ml-pipelines-v2"
        if (v2 / "build_master_datasets.py").exists():
            return v2
        p = p.parent
    raise FileNotFoundError("Could not find ml-pipelines-v2 — open from repo or set cwd to ml-pipelines-v2/")

ML_DIR = _find_ml_dir()
DATA_DIR = ML_DIR / "data"
MODEL_DIR = ML_DIR / "models"
MODEL_DIR.mkdir(exist_ok=True)
BUILDER = ML_DIR / "build_master_datasets.py"
if BUILDER.exists() and (
    not (DATA_DIR / "donor_master.csv").exists()
    or not (DATA_DIR / "resident_master.csv").exists()
):
    subprocess.run([sys.executable, str(BUILDER)], check=True)

df = pd.read_csv(DATA_DIR / 'resident_master.csv', low_memory=False)
TARGET = 'reintegration_ready'
print(df.shape); df[TARGET].value_counts()


In [ ]:
num_cols = ['avg_health_score_trend','avg_education_progress','incident_frequency','progress_noted_rate',
            'counseling_session_count','days_in_program']
cat_cols = ['initial_risk_level']
flags = ['sub_cat_trafficked','sub_cat_physical_abuse','sub_cat_sexual_abuse']
for f in flags:
    df[f] = df[f].astype(str).str.lower().eq('true').astype(int)
eda = df[num_cols+cat_cols+flags+[TARGET]]
print(eda.describe())
print(eda.isna().sum())


In [ ]:
for c in num_cols:
    df[c].hist(bins=20); plt.title(c); plt.show()
sns.heatmap(df[num_cols+[TARGET]].corr(), annot=True, fmt='.2f'); plt.show()
ct = pd.crosstab(df['initial_risk_level'].fillna('NA'), df[TARGET])
print(stats.chi2_contingency(ct)[:2])


In [ ]:
FEATURES = num_cols + cat_cols + flags
m = df[FEATURES + [TARGET]].dropna(subset=[TARGET])
for c in num_cols:
    m[c] = m[c].fillna(m[c].median())
m['initial_risk_level'] = m['initial_risk_level'].fillna('Unknown').astype(str)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
X = m[FEATURES]
y = m[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
num_f = num_cols + flags
cat_f = cat_cols
numeric_t = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
categorical_t = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
preprocess = ColumnTransformer([('num', numeric_t, num_f), ('cat', categorical_t, cat_f)])


## 3. Modeling & Feature Selection


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
pipe_lr = Pipeline([('prep', preprocess), ('clf', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42))])
pipe_rf = Pipeline([('prep', preprocess), ('clf', RandomForestClassifier(200, max_depth=6, random_state=42, class_weight='balanced'))])
pipe_lr.fit(X_train, y_train)
pipe_rf.fit(X_train, y_train)
print('LR AUC', roc_auc_score(y_test, pipe_lr.predict_proba(X_test)[:,1]))
print('RF AUC', roc_auc_score(y_test, pipe_rf.predict_proba(X_test)[:,1]))


In [ ]:
try:
    import shap
    # TreeExplainer on forest inner model — use sample for speed
    prep = pipe_rf.named_steps['prep']
    Xtr = prep.transform(X_train)
    explainer = shap.TreeExplainer(pipe_rf.named_steps['clf'])
    shap.summary_plot(explainer.shap_values(Xtr[:200]), feature_names=prep.get_feature_names_out(), show=False)
    plt.tight_layout(); plt.show()
except Exception as e:
    print('SHAP optional:', e)


In [ ]:
imp = pipe_rf.named_steps['clf'].feature_importances_
names = pipe_rf.named_steps['prep'].get_feature_names_out()
pd.Series(imp, index=names).sort_values(ascending=False).head(12)


## 4. Evaluation & Interpretation


In [ ]:
from sklearn.metrics import classification_report, RocCurveDisplay, confusion_matrix
import matplotlib.pyplot as plt
proba = pipe_rf.predict_proba(X_test)[:,1]
print(classification_report(y_test, pipe_rf.predict(X_test), digits=3))
fig, ax = plt.subplots(figsize=(5,4))
RocCurveDisplay.from_predictions(y_test, proba, ax=ax)
plt.show()
print(confusion_matrix(y_test, pipe_rf.predict(X_test)))


## 5. Causal / Relationship Analysis

**EXPLANATORY** sklearn **LogisticRegression** on train data — **associations** only; case management has many unobserved confounders (family, legal, external services).


In [ ]:
from sklearn.linear_model import LogisticRegression
Xe = pd.get_dummies(X_train.copy(), columns=['initial_risk_level'], drop_first=True)
Xe = Xe.apply(pd.to_numeric, errors='coerce').fillna(0.0)
lr_ex = LogisticRegression(max_iter=3000, class_weight='balanced', random_state=42, solver='lbfgs')
lr_ex.fit(Xe, y_train)
coef = pd.Series(lr_ex.coef_[0], index=Xe.columns)
print(np.exp(coef).sort_values(ascending=False).head(15))


## 6. Deployment Notes

Save **Random Forest pipeline** (if best) to `reintegration_model.sav`. **UI:** readiness **score** on **resident detail / Insights** page (replacing mock data).


In [ ]:
import joblib
import numpy as np
from sklearn.metrics import roc_auc_score
final = pipe_rf if roc_auc_score(y_test, pipe_rf.predict_proba(X_test)[:,1]) >= roc_auc_score(y_test, pipe_lr.predict_proba(X_test)[:,1]) else pipe_lr
joblib.dump(final, MODEL_DIR / 'reintegration_model.sav')
m = joblib.load(MODEL_DIR / 'reintegration_model.sav')
print('sample proba', m.predict_proba(X_test.iloc[:1]))
